In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# If needed, install tiktoken:
# Install with: pip install tiktoken"
import tiktoken

# A tiny helper for readability in prints
def show(name, x):
    print(f"{name}: shape={tuple(x.shape)}, dtype={x.dtype}, device={x.device}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Lecture 1: Practical 2 -- Towards Building A Model

In Practical 1, we looked at how PyTorch Tensors, how to manipulate them and their raw memory storage.

To build a Neural Network (like our LLM), we need two additional abstractions:
1. **Autograd (`requires_grad`)**: A way to automatically compute derivatives (gradients) using the chain rule.
2. **Modules (`nn.Module`)**: A way to group and manage stateful tensors (weights and biases).

We will build the foundational blocks of a GPT model: Multi-Head Attention and Transformer Block

## A) Parameters and Autograd
### Key ideas
- A tensor participates in autograd if `requires_grad=True`.
- `nn.Parameter` is just a tensor **registered** as a learnable parameter of an `nn.Module`.
- Calling `loss.backward()` computes gradients and stores them in `param.grad`.
- Gradients **accumulate** by default → you usually call `optimizer.zero_grad()` (or `param.grad = None`) each step.

We’ll start with the smallest possible example: one scalar parameter.


In [2]:
# One scalar parameter: y_hat = w * x, loss = (y_hat - y)^2
w = nn.Parameter(torch.tensor(2.0, device=device))
x = torch.tensor(3.0, device=device)
y = torch.tensor(10.0, device=device)

y_hat = w * x
loss = (y_hat - y).pow(2)

print("w:", w.item(), "y_hat:", y_hat.item(), "loss:", loss.item())

loss.backward()
print("dw:", w.grad.item())

# A manual SGD step
lr = 0.1
with torch.no_grad():
    w -= lr * w.grad
    w.grad = None  # reset

print("w after SGD step:", w.item())

w: 2.0 y_hat: 6.0 loss: 16.0
dw: -24.0
w after SGD step: 4.400000095367432


In [3]:
# Parameters inside a Module: Linear regression
lin = nn.Linear(4, 1, bias=True).to(device)

print("Named parameters:")
for n, p in lin.named_parameters():
    print(" ", n, tuple(p.shape), "requires_grad=", p.requires_grad)

X = torch.randn(8, 4, device=device)      # batch=8
y = torch.randn(8, 1, device=device)

pred = lin(X)
loss = F.mse_loss(pred, y)
loss.backward()

print("\nGradient norms:")
for n, p in lin.named_parameters():
    print(" ", n, float(p.grad.norm()))


Named parameters:
  weight (1, 4) requires_grad= True
  bias (1,) requires_grad= True

Gradient norms:
  weight 1.2559733390808105
  bias 1.23886239528656


## B) Tokenization + Embedding

Language models cannot read text. They read numbers. 
A tokenizer converts text into integer IDs (e.g., `"apple" -> 412`). 
An Embedding layer converts integer IDs into dense continuous vectors (e.g., `412 -> [0.1, -0.4, 0.8, ...]`).
- Tokenizer: `text -> List[int]`
- Embedding: `ids [B,T] -> vectors [B,T,d_model]`

`nn.Embedding(V, d_model)` stores a matrix `W ∈ R^{V × d_model}`.
Given IDs, it returns rows of that matrix. It is equivalent to multiplying a (huge) one-hot vector by `W`, but *much faster*.

We’ll use **tiktoken** (OpenAI’s fast BPE tokenizer). If you don’t have it installed, install it once.

In [4]:
tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello world! こんにちは 🌍"
ids = tokenizer.encode(text)
print("text:", text)
print("ids:", ids)
print("num tokens:", len(ids))

# Make a tiny batch: pad to same length
B = 2
T = 8
pad_id = tokenizer.eot_token  # GPT-2 uses end-of-text as a convenient pad
batch = torch.full((B, T), pad_id, dtype=torch.long)
batch[0, :min(T, len(ids))] = torch.tensor(ids[:T])

show("batch token ids", batch)


text: Hello world! こんにちは 🌍
ids: [15496, 995, 0, 23294, 241, 22174, 28618, 2515, 94, 31676, 12520, 234, 235]
num tokens: 13
batch token ids: shape=(2, 8), dtype=torch.int64, device=cpu


## C) Multi-Head Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

However, if you look at PyTorch's native `nn.MultiheadAttention` (which we use in our `model.py`), you won't find three separate `nn.Linear` layers for $Q$, $K$, and $V$. 

**Why? Kernel Fusion.**
Executing three separate matrix multiplications (`x @ W_q`, `x @ W_k`, `x @ W_v`) means reading `x` from GPU memory three separate times. Memory bandwidth is the primary bottleneck in modern LLMs. PyTorch optimizes this by concatenating the weights into a single massive parameter called `in_proj_weight` (shape: `[3 * embed_dim, embed_dim]`). It computes the projection all at once, and then splits the result.

Let's build a clean, readable implementation of Multi-Head Attention that mimics this optimization and serves as a **drop-in replacement** for PyTorch's native layer.

## 3) Multi-Head Self-Attention (MHA)

$$ \mathrm{Attention}(Q,K,V)=\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V $$

**Input:** `x ∈ R^{B×T×D}`  
- `B` = batch size  
- `T` = sequence length  
- `D` = embedding dimension  

### Step-by-step (shape tracking)

1) **Linear projections**
- `Q = x W_Q`, `K = x W_K`, `V = x W_V`  each in `R^{B×T×D}`

2) **Split channels into heads**
Let `H` = number of heads and `d = D/H`.
- `Q → R^{B×H×T×d}`, `K → R^{B×H×T×d}`, `V → R^{B×H×T×d}`

3) **Attention scores**
- `scores = (Q @ K.transpose(-2,-1)) / sqrt(d)` in `R^{B×H×T×T}`

Then apply a **causal mask** so position `t` can’t see future positions `> t`.

4) **Softmax**
- `A = softmax(scores)` over the **last dimension** (keys)

5) **Weighted sum**
- `Y = A @ V` in `R^{B×H×T×d}`

6) **Merge heads + output projection**
- merge heads: `Y → R^{B×T×D}`
- output projection: `Y W_O`

---

### Exercise: implement `SimpleMultiheadAttention`

Goal: implement a minimal MHA that is easy to read and matches the **output shapes** of `nn.MultiheadAttention`.

**For this practical, you can assume the common GPT case:**
- *self-attention*: `query == key == value`
- you only really need `is_causal=True` (ignore other masks at first)

**Implementation plan**
- TODO(1) project `query/key/value` → `Q,K,V`
- TODO(2) reshape to heads
- TODO(3) compute `scores`
- TODO(4) apply causal mask (and optionally padding / attn masks)
- TODO(5) softmax (and dropout)
- TODO(6) weighted sum, merge heads, output projection

Tip: implement it in small steps and run the sanity checks after each step.


In [5]:
def causal_attention_mask(seq_len, device):
    """Upper-triangular boolean mask; True means 'do not attend'."""
    return torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device), diagonal=1)

def scaled_dot_product_attention(Q, K, V, causal_mask=None):
    """
    Attention(Q, K, V) = softmax( Q K^T / sqrt(d_k) ) · V

    Q, K, V : (B, H, L, d_k)
    returns   (B, H, L, d_k),  (B, H, L, L)
    """
    # Q K^T → (B, H, L, L) score matrix; scale to stabilise gradients
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Q.size(-1))

    # mask future positions: set their scores to -inf so softmax → 0
    if causal_mask is not None:
        scores = scores.masked_fill(causal_mask, float('-inf'))

    # each row becomes a probability distribution over positions
    weights = F.softmax(scores, dim=-1)           # (B, H, L, L)

    # weighted sum of value vectors: (B, H, L, L) · (B, H, L, d_k) → (B, H, L, d_k)
    return torch.matmul(weights, V), weights
    
class MultiHeadAttentionFromScratch(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.H, self.d_k = num_heads, embed_dim // num_heads
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)

        for proj in (self.W_q, self.W_k, self.W_v, self.W_o):
            nn.init.xavier_uniform_(proj.weight)
            nn.init.zeros_(proj.bias)

    def _split_heads(self, t, B, L):
        # (B, L, D) → (B, H, L, d_k)
        return t.view(B, L, self.H, self.d_k).transpose(1, 2)

    def forward(self, x, causal_mask=None):
        B, L, D = x.shape

        # TODO 1: project x through W_q, W_k, W_v and split into heads
        #         Expected shape after this step: (B, H, L, d_k) for each
        Q = ...
        K = ...
        V = ...

        # TODO 2: call scaled_dot_product_attention(Q, K, V, causal_mask)
        #         It returns (attn_out, weights); you only need attn_out here
        attn_out = ...

        # TODO 3: merge the H heads back into one tensor of shape (B, L, D)
        #         then apply the output projection W_o
        return None

In [6]:
B, L, D, H = 2, 6, 32, 4
x_test = torch.randn(B, L, D)
mask   = causal_attention_mask(L, x_test.device)

student = MultiHeadAttentionFromScratch(embed_dim=D, num_heads=H)
ref     = nn.MultiheadAttention(embed_dim=D, num_heads=H, dropout=0.0, batch_first=True)

# Sync weights: copy student's separate W_q/W_k/W_v into ref's fused in_proj_weight
with torch.no_grad():
    ref.in_proj_weight.copy_(torch.cat([student.W_q.weight, student.W_k.weight, student.W_v.weight]))
    ref.in_proj_bias.copy_(torch.cat([student.W_q.bias, student.W_k.bias, student.W_v.bias]))
    ref.out_proj.weight.copy_(student.W_o.weight)
    ref.out_proj.bias.copy_(student.W_o.bias)

out     = student(x_test, causal_mask=mask)
out_ref, _ = ref(x_test, x_test, x_test,
                 attn_mask=mask.float().masked_fill(mask, float('-inf')),
                 need_weights=False)

assert out.shape == (B, L, D), f"Shape mismatch: {out.shape}"
assert torch.allclose(out, out_ref, atol=1e-5), f"Max diff: {(out - out_ref).abs().max():.2e}"
print(f"[PASS] shape {tuple(out.shape)}, max diff {(out - out_ref).abs().max():.2e}")


AttributeError: 'NoneType' object has no attribute 'shape'

## 4) Transformer block

A decoder-only Transformer block with **post-norm** looks like:

```
x = LN(x + dropout(Attn(x)))     # causal self-attention + dropout + residual + norm
x = LN(x + dropout(MLP(x)))      # feed-forward + dropout + residual + norm
```

Causality is enforced by passing a boolean `attn_mask` to MHA (True = ignore that position):

```python
mask = causal_attention_mask(T, device)   # shape [T, T], upper-triangular True
attn_out, _ = mha(x, x, x, attn_mask=mask, need_weights=False)
```

### Exercise: implement `TransformerBlock`

**Layers defined in `__init__`:**
- `self.mha` — `nn.MultiheadAttention(embed_dim, num_heads, dropout=0.0, batch_first=True)`
- `self.ln1`, `self.ln2` — `nn.LayerNorm(embed_dim, eps=1e-6)`
- `self.lin1` — `nn.Linear(embed_dim, ff_dim)` (expands to wider hidden size)
- `self.lin2` — `nn.Linear(ff_dim, embed_dim)` (projects back)


In [ ]:
class TransformerBlock(nn.Module):
    """A minimal Transformer block that matches the structure in model.py (post-norm).

    Structure:
      x -> MHA (causal self-attn) -> dropout -> +residual -> LN
        -> FFN (Linear->ReLU->Linear) -> dropout -> +residual -> LN
    """
    def __init__(self,
                 embed_dim: int,
                 num_heads: int,
                 ff_dim: int,
                 *,
                 dropout_rate: float = 0.1):

        super().__init__()

        self.dropout_rate = dropout_rate

        # Match JAX: no dropout inside attention weights
        self.mha = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=0.0,
            batch_first=True
        )
        nn.init.xavier_uniform_(self.mha.in_proj_weight)
        nn.init.zeros_(self.mha.in_proj_bias)
        nn.init.xavier_uniform_(self.mha.out_proj.weight)
        nn.init.zeros_(self.mha.out_proj.bias)

        self.ln1 = nn.LayerNorm(embed_dim, eps=1e-6)
        nn.init.ones_(self.ln1.weight)
        nn.init.zeros_(self.ln1.bias)

        self.lin1 = nn.Linear(embed_dim, ff_dim)
        nn.init.xavier_uniform_(self.lin1.weight)
        nn.init.zeros_(self.lin1.bias)

        self.lin2 = nn.Linear(ff_dim, embed_dim)
        nn.init.xavier_uniform_(self.lin2.weight)
        nn.init.zeros_(self.lin2.bias)


        self.ln2 = nn.LayerNorm(embed_dim, eps=1e-6)
        nn.init.ones_(self.ln2.weight)
        nn.init.zeros_(self.ln2.bias)

    def forward(self, inputs, training: bool = False):
        _, seq_len, _ = inputs.shape

        mask = causal_attention_mask(seq_len, inputs.device)

        # PyTorch requires q,k,v; JAX defaults k=v=q
        attention_output, _ = self.mha(
            inputs, inputs, inputs,
            attn_mask=mask,
            need_weights=False
        )

        return ...


In [ ]:
B, T, C, H = 2, 6, 32, 4
ff = 4 * C
x = torch.randn(B, T, C, device=device, requires_grad=True)

blk = TransformerBlock(embed_dim=C, num_heads=H, ff_dim=ff, dropout_rate=0.0).to(device)
y = blk(x)

assert(y.shape == (B, T, C))

loss = y.pow(2).mean()
loss.backward()
print("OK: backward ran. grad norm:", float(x.grad.norm()))

AttributeError: 'NoneType' object has no attribute 'shape'


# Token + Position Embeddings

In a GPT model, a token's meaning depends not just on *what* it is, but *where* it is in the sequence.
Therefore, the input to our Transformer block is the sum of two embeddings:
1. **Action/Token Embedding:** What the word is (Vocab Size $\times$ Embed Dim)
2. **Sequence/Position Embedding:** Where the word is (Max Sequence Length $\times$ Embed Dim)

In [ ]:
class SimpleGPT(nn.Module):
    def __init__(self, vocab_size: int, context_length: int, embed_dim: int,
                 num_layers: int, num_heads: int, ff_dim: int, dropout_rate: float = 0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.embed_dim = embed_dim

        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_length, embed_dim)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim=embed_dim, num_heads=num_heads, ff_dim=ff_dim, dropout_rate=dropout_rate)
            for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size, bias=False)

    def forward(self, idx: torch.Tensor):
        # idx: [B,T] integer token ids
        B, T = idx.shape
        assert T <= self.context_length

        pos = torch.arange(T, device=idx.device)           # [T]
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]   # [B,T,C]

        for blk in self.blocks:
            x = blk(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)                           # [B,T,V]
        return logits


In [ ]:
# --- toy run (once your block + MHA are implemented) ---
V = tokenizer.n_vocab
C = 32
T = 16
model = SimpleGPT(vocab_size=V, context_length=T, embed_dim=C, num_layers=2, num_heads=4, ff_dim=4*C, dropout_rate=0.0).to(device)

# Batch of tokens
idx = batch[:, :T]  # [B,T]
logits = model(idx)
show("logits", logits)

# Next-token prediction loss: shift by 1
targets = idx[:, 1:].contiguous()
logits_for_loss = logits[:, :-1, :].contiguous()

loss = F.cross_entropy(logits_for_loss.reshape(-1, V), targets.reshape(-1))
print("loss:", float(loss))

loss.backward()
print("OK: backprop through SimpleGPT ran.")

AttributeError: 'NoneType' object has no attribute 'shape'